Setup Kaggle n Git

In [1]:
import os
import warnings
warnings.filterwarnings("ignore")

DATA_PATH = "/kaggle/input/datasets/williamscott701/memotion-dataset-7k/memotion_dataset_7k"

print(os.listdir(DATA_PATH))

['reference.csv', 'labels_pd_pickle', 'labels.xlsx', 'reference.xlsx', 'images', 'labels.csv', 'reference_df_pickle']


In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
GITHUB_TOKEN = user_secrets.get_secret("Hate Meme Repo Access Token")
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

In [3]:
!git clone https://{GITHUB_TOKEN}@github.com/Parthivendra/multimodal-hate-meme-detection.git
%cd multimodal-hate-meme-detection

Cloning into 'multimodal-hate-meme-detection'...
remote: Enumerating objects: 110, done.
remote: Counting objects: 100% (110/110), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 110 (delta 68), reused 66 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (110/110), 28.75 KiB | 2.87 MiB/s, done.
Resolving deltas: 100% (68/68), done.
/kaggle/working/multimodal-hate-meme-detection


In [4]:
!git config --global user.email "parthivendra2004@gmail.com"
!git config --global user.name "Parthivendra Singh"

In [5]:
!git pull

Already up to date.


In [6]:
import sys
sys.path.append("/kaggle/working")
sys.path.append("/kaggle/working/multimodal-hate-meme-detection")

In [7]:
# reload custom modules in case of remote update via git or otherwise
import importlib
import src.dataset
import src.model_text
import src.model_image
import src.train
import src.model_multimodal
importlib.reload(src.dataset)
importlib.reload(src.model_text)
importlib.reload(src.model_image)
importlib.reload(src.train)
importlib.reload(src.model_multimodal)

<module 'src.model_multimodal' from '/kaggle/working/multimodal-hate-meme-detection/src/model_multimodal.py'>

Code

In [8]:
from huggingface_hub import login
login(HF_TOKEN)

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [9]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [10]:
# import and validate dataset
import pandas as pd
import os

df = pd.read_csv(f"{DATA_PATH}/labels.csv", index_col=0)

# check for any corrupt/non-existent paths
IMG_DIR = f"{DATA_PATH}/images"
valid_images = set(os.listdir(IMG_DIR))

df = df[df["image_name"].isin(valid_images)].reset_index(drop=True)
print("Clean dataset size:", len(df))

Clean dataset size: 6992


In [11]:
from src.dataset import MemeDataset
from torch.utils.data import DataLoader

dataset = MemeDataset(
    df=df,
    img_dir=f"{DATA_PATH}/images",
    tokenizer=tokenizer,
    transform=transform
)

loader = DataLoader(dataset, batch_size=4, shuffle=True)

In [12]:
batch = next(iter(loader))

print(batch["input_ids"].shape)
print(batch["image"].shape)
print(batch["label"])

torch.Size([4, 128])
torch.Size([4, 3, 224, 224])
tensor([1, 1, 0, 0])


In [13]:
from src.model_text import TextModel

model = TextModel(num_classes=4)

batch = next(iter(loader))

outputs = model(
    input_ids=batch["input_ids"],
    attention_mask=batch["attention_mask"]
)

print(outputs.shape)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([4, 4])


In [14]:
print(outputs)

tensor([[ 0.1209, -0.3674, -0.0065,  0.1389],
        [ 0.1482, -0.1958,  0.5649,  0.3570],
        [-0.1872, -0.0085,  0.4630,  0.2682],
        [-0.2067,  0.1551,  0.5952, -0.0116]], grad_fn=<AddmmBackward0>)


In [15]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")
print(device)

cuda


In [16]:
model = TextModel(num_classes=4)
model.to(device)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


TextModel(
  (bert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
            (lin1): Linear(in_fe

In [17]:
import torch.nn as nn
from torch.optim import AdamW

criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=2e-5)

In [18]:
# # Testing src/train.py

# # run with kaggle T4 not with P100

# from src.train import train_one_epoch

# EPOCHS = 1  # small for now

# for epoch in range(EPOCHS):
#     loss = train_one_epoch(model, loader, optimizer, criterion, device)
#     print(f"Epoch {epoch+1} Loss: {loss:.4f}")

In [19]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["offensive"]
)

In [20]:
train_dataset = MemeDataset(
    df=train_df,
    img_dir="/kaggle/input/memotion-dataset/images",
    tokenizer=tokenizer,
    transform=transform
)

val_dataset = MemeDataset(
    df=val_df,
    img_dir="/kaggle/input/memotion-dataset/images",
    tokenizer=tokenizer,
    transform=transform
)

from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4)

In [21]:
from src.train import train_one_epoch, evaluate

EPOCHS = 2 # low for start

for epoch in range(EPOCHS):
    loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    acc, f1 = evaluate(model, val_loader, device)

    print(f"\nEpoch {epoch+1}")
    print(f"Loss: {loss:.4f}")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Score: {f1:.4f}")

100%|██████████| 1399/1399 [01:28<00:00, 15.86it/s]



Epoch 1
Loss: 1.1862
Accuracy: 0.3710
F1 Score: 0.2008


100%|██████████| 1399/1399 [01:32<00:00, 15.18it/s]



Epoch 2
Loss: 1.1724
Accuracy: 0.3867
F1 Score: 0.3026


In [22]:
torch.save(model.state_dict(), "/kaggle/working/multimodal-hate-meme-detection/text_model.pt")

In [23]:
!pwd

/kaggle/working/multimodal-hate-meme-detection


In [24]:
from src.model_text import TextModel

model = TextModel(num_classes=4)
model.load_state_dict(torch.load("text_model.pt"))
model.to(device)
model.eval()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


TextModel(
  (bert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
            (lin1): Linear(in_fe

In [25]:
# Predict Text Function and Test Predictions

def predict_text(text, model, tokenizer, device):
    model.eval()

    encoding = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(input_ids, attention_mask)
        pred = torch.argmax(outputs, dim=1).item()

    return pred

id2label = {
    0: "not_offensive",
    1: "slight",
    2: "very_offensive",
    3: "hateful_offensive"
}

text = "I hate you so much"

pred = predict_text(text, model, tokenizer, device)

print("Prediction:", id2label[pred])

Prediction: not_offensive


In [26]:
from src.model_image import ImageModel

image_model = ImageModel(num_classes=4).to(device)

batch = next(iter(train_loader))

outputs = image_model(batch["image"].to(device))

print(outputs.shape)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 138MB/s]


torch.Size([4, 4])


In [27]:
def train_image(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    for batch in loader:
        images = batch["image"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [28]:
# FIX 4: Sanity Check (Before Training)
batch = next(iter(train_loader))
outputs = image_model(batch["image"].to(device))
print(torch.argmax(outputs, dim=1))
# If all predictions are same class -> confirmed collapse

tensor([2, 2, 2, 2], device='cuda:0')


In [29]:
from tqdm import tqdm
import torch.nn as nn
from torch.optim import AdamW
from src.train import evaluate
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch

classes = np.unique(train_df["offensive"])
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["offensive"]
)

print('Computed Class Weights:', weights)

label_map = {
    "not_offensive": 0,
    "slight": 1,
    "very_offensive": 2,
    "hateful_offensive": 3
}

weight_tensor = torch.tensor([
    weights[list(classes).index("not_offensive")],
    weights[list(classes).index("slight")],
    weights[list(classes).index("very_offensive")],
    weights[list(classes).index("hateful_offensive")]
], dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=weight_tensor)
optimizer = AdamW(image_model.parameters(), lr=1e-5)

EPOCHS = 2

for epoch in tqdm(range(EPOCHS), desc="Training Progress"):
    loss = train_image(image_model, train_loader, optimizer, criterion, device)
    acc, f1 = evaluate(image_model, val_loader, device, mode="image")

    tqdm.write(f"\nEpoch {epoch+1}")
    tqdm.write(f"Loss: {loss:.4f}")
    tqdm.write(f"Accuracy: {acc:.4f}")
    tqdm.write(f"F1 Score: {f1:.4f}")


Computed Class Weights: [7.89971751 0.64435484 0.67450555 1.19202899]


Training Progress:  50%|█████     | 1/2 [00:17<00:17, 17.65s/it]


Epoch 1
Loss: 1.3315
Accuracy: 0.3881
F1 Score: 0.2171


Training Progress: 100%|██████████| 2/2 [00:35<00:00, 17.52s/it]


Epoch 2
Loss: 1.3059
Accuracy: 0.3881
F1 Score: 0.2171


In [30]:
from src.model_multimodal import MultimodalModel

mm_model = MultimodalModel(num_classes=4).to(device)

batch = next(iter(train_loader))

outputs = mm_model(
    batch["input_ids"].to(device),
    batch["attention_mask"].to(device),
    batch["image"].to(device)
)

print(outputs.shape)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([4, 4])


In [31]:
def train_multimodal(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        images = batch["image"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        outputs = model(input_ids, attention_mask, images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [32]:
from src.train import evaluate

mm_model = MultimodalModel(num_classes=4).to(device)

optimizer = AdamW(mm_model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss(weight=weight_tensor)

EPOCHS = 3

for epoch in range(EPOCHS):
    loss = train_multimodal(mm_model, train_loader, optimizer, criterion, device)
    acc, f1 = evaluate(mm_model, val_loader, device, mode="multimodal")

    print(f"\nEpoch {epoch+1}")
    print(f"Loss: {loss:.4f}")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Score: {f1:.4f}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Epoch 1
Loss: 1.3194
Accuracy: 0.3710
F1 Score: 0.2008

Epoch 2
Loss: 1.3082
Accuracy: 0.3681
F1 Score: 0.2154

Epoch 3
Loss: 1.2400
Accuracy: 0.3109
F1 Score: 0.2935


In [33]:
import torch
from PIL import Image

def predict_meme(text, image_path, model, tokenizer, transform, device):
    model.eval()

    # ---- TEXT ----
    encoding = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    # ---- IMAGE ----
    image = Image.open(image_path).convert("RGB")
    image = transform(image).unsqueeze(0).to(device)

    # ---- PREDICT ----
    with torch.no_grad():
        outputs = model(input_ids, attention_mask, image)
        pred = torch.argmax(outputs, dim=1).item()

    return pred

In [34]:
id2label = {
    0: "not_offensive",
    1: "slight",
    2: "very_offensive",
    3: "hateful_offensive"
}

In [35]:
text = "Understandable, have a great day"

image_path = "/kaggle/input/datasets/williamscott701/memotion-dataset-7k/memotion_dataset_7k/images/image_1009.png"

pred = predict_meme(
    text,
    image_path,
    mm_model,
    tokenizer,
    transform,
    device
)

print("Prediction:", id2label[pred])

Prediction: very_offensive


In [36]:
!ls /kaggle/input/datasets/williamscott701/memotion-dataset-7k/memotion_dataset_7k/images/image_1007.jpg

/kaggle/input/datasets/williamscott701/memotion-dataset-7k/memotion_dataset_7k/images/image_1007.jpg
